In [ ]:

from analysis import run_full_analysis

# Run full analysis using detection module
image_path = "Merged-CCCP.tif"  # Replace with your file
output_dir = "outputs"
run_full_analysis(image_path=image_path, output_dir=output_dir, mito_channel=0, lyso_channel=1)

# Outputs will include:
# - Detection_Outlines_Frame0.png
# - Channel_0_Lysosomes_Frame0.png
# - Channel_1_Mitochondria_Frame0.png


# AutoMorphoTrack Full Package Code\n\nThis notebook contains the full code for the AutoMorphoTrack package, including detection, morphology classification, motion analysis, colocalization, and example usage.

In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk
# Detection module\nimport numpy as np\nfrom skimage.filters import threshold_isodata\nfrom skimage.measure import label, regionprops\nfrom skimage.morphology import remove_small_objects\n\ndef threshold_and_detect(image_stack, min_size=5):\n    thresh_stack = np.array([frame > threshold_isodata(frame) for frame in image_stack])\n    masks = [label(remove_small_objects(frame, min_size=min_size)) for frame in thresh_stack]\n    props_list = [regionprops(mask) for mask in masks]\n    return props_list

In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk
# Morphology module\nimport pandas as pd\n\ndef classify_mitochondria_morphology(props_list):\n    morphology_per_frame = []\n    for i, props in enumerate(props_list):\n        elongated = 0\n        punctate = 0\n        for prop in props:\n            area = prop.area\n            eccentricity = prop.eccentricity\n            if area >= 0.025 and eccentricity >= 0.85:\n                elongated += 1\n            elif area >= 0.03 and eccentricity <= 0.7:\n                punctate += 1\n        morphology_per_frame.append({'Frame': i, 'Elongated': elongated, 'Punctate': punctate})\n    return pd.DataFrame(morphology_per_frame)

In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk
# Motion module\nfrom scipy.spatial import distance\n\ndef calculate_motion(props_list):\n    tracks = [[prop.centroid for prop in props] for props in props_list]\n    displacements = []\n    velocities = []\n    for i in range(1, len(tracks)):\n        prev_centroids = tracks[i - 1]\n        curr_centroids = tracks[i]\n        frame_displacements = []\n        for curr in curr_centroids:\n            if not prev_centroids:\n                frame_displacements.append(0)\n                continue\n            dists = [distance.euclidean(curr, prev) for prev in prev_centroids]\n            frame_displacements.append(np.min(dists))\n        displacements.append(np.mean(frame_displacements) if frame_displacements else 0)\n        velocities.append(np.mean(frame_displacements) if frame_displacements else 0)\n    return pd.DataFrame({'Frame': range(1, len(displacements) + 1), 'Mean Displacement': displacements, 'Mean Velocity': velocities})

In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk
# Colocalization module\nfrom scipy.stats import pearsonr\n\ndef calculate_colocalization_pearson(stack_mito, stack_lyso):\n    return [pearsonr(mito.flatten(), lyso.flatten())[0] for mito, lyso in zip(stack_mito, stack_lyso)]

In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk
# Full stack processing utility\nimport tifffile\nimport os\n\ndef process_stack_all(path, treatment):\n    stack_name = os.path.basename(path).replace('.tif', '')\n    image_stack = tifffile.imread(path)\n    mito_stack = image_stack[:, 1, :, :]\n    lyso_stack = image_stack[:, 0, :, :]\n\n    props_list = threshold_and_detect(mito_stack)\n    df_morphology = classify_mitochondria_morphology(props_list)\n    df_morphology['Stack'] = stack_name\n    df_morphology['Treatment'] = treatment\n\n    df_motion = calculate_motion(props_list)\n    df_motion['Stack'] = stack_name\n    df_motion['Treatment'] = treatment\n\n    colocalization_scores = calculate_colocalization_pearson(mito_stack, lyso_stack)\n    \n    return df_morphology, df_motion, colocalization_scores

## Example Usage

In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk
# Example analysis\n# df_morphology, df_motion, colocalization_scores = process_stack_all('example_file.tif', 'PFF+DMSO')\n\n# print(df_morphology)\n# print(df_motion)\n# print(colocalization_scores)

In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk

# Save labeled image of mitochondrial morphology from first frame
from morphology import save_mito_morphology_image

# Assuming variables `first_frame`, `labeled_mask`, and `morphology_labels` are defined earlier in the notebook
save_mito_morphology_image(
    first_frame=first_frame,
    labeled_mask=labeled_mask,
    morphology_labels=morphology_labels,
    output_path="morphology_labeled_frame.png"
)


In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk

from detection import generate_frame0_detection_overlay

# Run after loading your TIFF and extracting channels
# Assuming `lysosome_stack` and `mitochondria_stack` are available
first_lyso = lysosome_stack[0].astype(float) / 255.0
first_mito = mitochondria_stack[0].astype(float) / 255.0

generate_frame0_detection_overlay(
    lyso_frame=first_lyso,
    mito_frame=first_mito,
    output_path="organelle_detection_overlay_frame0.png"
)


In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk

from detection import generate_frame0_contour_overlay

# Assuming lysosome_stack and mitochondria_stack already loaded
first_lyso = lysosome_stack[0].astype(float) / 255.0
first_mito = mitochondria_stack[0].astype(float) / 255.0

generate_frame0_contour_overlay(
    lyso_frame=first_lyso,
    mito_frame=first_mito,
    output_path="organelle_detection_overlay_frame0_contours.png"
)


In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk

from visualization import visualize_colocalization_frame0
import tifffile

# Load image stack
image_stack = tifffile.imread(path)
mito_stack = image_stack[:, 1, :, :]
lyso_stack = image_stack[:, 0, :, :]

# Visualize colocalization on Frame 0
visualize_colocalization_frame0(mito_stack, lyso_stack)


In [ ]:
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.morphology import dilation, disk

from skimage.measure import label, regionprops
from visualization import draw_mitochondrial_morphology, draw_lysosome_count

# Extract frame 0 and segment
mito_mask0 = mito_stack[0] > threshold_otsu(gaussian_filter(mito_stack[0], sigma=1))
lyso_mask0 = lyso_stack[0] > threshold_otsu(gaussian_filter(lyso_stack[0], sigma=1))

mito_props0 = regionprops(label(mito_mask0))
lyso_props0 = regionprops(label(lyso_mask0))

# Draw separately
draw_mitochondrial_morphology(mito_mask0, mito_props0, filename="frame0_mitochondria_morphology_only.png")
draw_lysosome_count(lyso_mask0, lyso_props0, filename="frame0_lysosome_count_only.png")


## 🔄 Visualize Multicolor Organelle Tracks (Frame 0)
This section overlays lysosomal and mitochondrial movement trajectories as colored lines on the raw image from Frame 0.

In [ ]:

# Visualize multicolor trajectory tracks on frame 0
from tracking import visualize_multicolor_tracks

# Recreate object track dictionaries from previously run tracking data
from tracking import track_organelles  # this will internally call visualize_multicolor_tracks again
track_organelles(data, output_dir)


## 🧪 Extended Quantification Analyses

In [ ]:
# Import new analysis module
from analysis import compute_shape_descriptors, compute_directionality, compute_lyso_mito_distances

In [ ]:
# Compute shape descriptors and save plots
compute_shape_descriptors(data, output_dir)

In [ ]:
# Compute directionality index from tracking CSV and plot histogram
tracking_csv = f"{output_dir}/displacement_velocity_mito.csv"
compute_directionality(tracking_csv, f"{output_dir}/directionality_summary.csv", f"{output_dir}/directionality_hist.png", data['stack'][0], channel=1)

In [ ]:
# Compute minimum lysosome-to-mitochondria distances and plot histogram
compute_lyso_mito_distances(data, output_dir)

### Colocalization Montage
This cell generates a montage image summarizing colocalized pixels (cyan) across 5 selected frames.

### Enhanced Organelle Tracking
The following visualizations are now supported:
- Rainbow trails
- Arrowed trajectories
- Occupancy heatmaps
- Start/end path summaries
Each is overlaid on Frame 0 and exported as PNG.

In [ ]:

from morphology import classify_and_visualize_morphology_across_frames

# Apply relaxed morphology classification across all frames
# classify_and_visualize_morphology_across_frames(mito_labels_stack, mito_image_stack, output_dir)


In [ ]:

from detection import count_lysosomes_across_frames

# Example usage:
# lysosome_stack = stack[:, 1]  # Assuming channel 1 is lysosomes
# count_lysosomes_across_frames(lysosome_stack, output_dir)
